In [ ]:
import os
import random
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Dissertation"
Graz_path = PROJECT_DIR + "/datasets/GRAZPEDWRI-DX"
part1 = Path(Graz_path + "/images_part1")

Extract the zip file for images_part2 to images_part4

In [ ]:
import zipfile, os

base = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX"
parts = ["images_part2", "images_part3", "images_part4"]

for part in parts:
    zip_path = f"{base}/{part}.zip"
    extract_path = f"/content/{part}"
    if not os.path.exists(extract_path):
        print(f"Extracting {part}...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(extract_path)
        print(f"Done: {part}")
    else:
        print(f"{part} already extracted, skipping")

Extracting images_part2...
Done: images_part2
Extracting images_part3...
Done: images_part3
Extracting images_part4...
Done: images_part4


1. Load GRAZPEDWRI-DX metadata CSV and inspect it

In [ ]:
df = pd.read_csv(Graz_path +"/dataset.csv")
print(df["fracture_visible"].value_counts(dropna=False))
print(df["filestem"].head())

fracture_visible
1.0    13550
NaN     6777
Name: count, dtype: int64
0    0001_1297860395_01_WRI-L1_M014
1    0001_1297860435_01_WRI-L2_M014
2    0002_0354485735_01_WRI-R1_F012
3    0002_0354485759_01_WRI-R2_F012
4    0003_0662359226_01_WRI-R1_M011
Name: filestem, dtype: object


2. Filter dataset.csv rows to match images_part1 (superseded now — we're using all 4 parts)

In [ ]:
IMG_DIR = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/images_part1"
part1_files = os.listdir(IMG_DIR)
part1_stems = set(os.path.splitext(f)[0] for f in part1_files)
df_part1 = df[df["filestem"].isin(part1_stems)].copy()

In [ ]:
parts = ["images_part1", "images_part2", "images_part3", "images_part4"]

# map filestem -> actual full path, checking across all four locations
path_map = {}
for part in parts:
    if part == "images_part1":
        dir_path = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/images_part1"
    else:
        dir_path = f"/content/{part}"
    for fname in os.listdir(dir_path):
        stem = os.path.splitext(fname)[0]
        path_map[stem] = os.path.join(dir_path, fname)

df["image_path"] = df["filestem"].map(path_map)

missing = df["image_path"].isna().sum()
print(f"Rows with no matching image file: {missing}")

df_matched = df.dropna(subset=["image_path"]).copy()
print(f"Matched rows: {len(df_matched)}")

Rows with no matching image file: 0
Matched rows: 20327


3. Convert GRAZPEDWRI-DX rows → prompt/response examples

In [ ]:
def row_to_example(row):
    img_path = row["image_path"]  # already resolved in df_matched

    view = row['projection'] if pd.notna(row['projection']) else ""
    side = row['laterality'] if pd.notna(row['laterality']) else ""
    view_desc = f"{side} {view}".strip() or "wrist"

    findings = []
    has_fracture = pd.notna(row['fracture_visible']) and row['fracture_visible'] == 1.0

    if has_fracture:
        if pd.notna(row.get('ao_classification')):
            findings.append(f"a fracture (AO classification: {row['ao_classification']})")
        else:
            findings.append("a fracture")

    if pd.notna(row.get('osteopenia')) and row['osteopenia'] == 1.0:
        findings.append("osteopenia")
    if pd.notna(row.get('metal')) and row['metal'] == 1.0:
        findings.append("metal hardware")
    if pd.notna(row.get('cast')) and row['cast'] == 1.0:
        findings.append("a cast in place")

    uncertain = pd.notna(row.get('diagnosis_uncertain')) and row['diagnosis_uncertain'] == 1.0

    if findings:
        response = f"This {view_desc} pediatric wrist X-ray shows " + ", ".join(findings) + "."
        if uncertain:
            response += " The diagnosis is noted as uncertain."
    else:
        response = f"This {view_desc} pediatric wrist X-ray shows no fracture or significant abnormality."

    return {
        "image_path": img_path,
        "prompt": "Describe any fracture or abnormal findings in this pediatric wrist X-ray.",
        "response": response,
        "fracture_visible": has_fracture,
        "source": "grazpedwri"
    }

examples_full = df_matched.apply(row_to_example, axis=1).tolist()
print(f"Total GRAZPEDWRI-DX examples: {len(examples_full)}")

Total GRAZPEDWRI-DX examples: 20327


4. Convert Mendeley images → prompt/response examples

In [ ]:
MENDELEY_ROOT = "/content/drive/MyDrive/Dissertation/datasets/Bone_fracture_dataset/Original"
FRACTURE_TYPES = {
    "Simple Bone Fracture": "a simple (non-displaced) bone fracture",
    "Comminuted Bone Fracture": "a comminuted bone fracture, where the bone is broken into multiple fragments"
}

def build_mendeley_examples(root=MENDELEY_ROOT):
    examples = []
    for folder_name, description in FRACTURE_TYPES.items():
        folder_path = os.path.join(root, folder_name)
        for fname in os.listdir(folder_path):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            examples.append({
                "image_path": os.path.join(folder_path, fname),
                "prompt": "Describe any fracture or abnormal findings in this X-ray.",
                "response": f"This X-ray shows evidence of {description}.",
                "fracture_visible": True,
                "source": "mendeley"
            })
    return examples

mendeley_examples = build_mendeley_examples()
mendeley_ds = Dataset.from_list(mendeley_examples)

5. Merge into one HF Dataset

In [ ]:
from datasets import Dataset, concatenate_datasets

grazpedwri_ds_full = Dataset.from_list(examples_full)
mendeley_ds = Dataset.from_list(mendeley_examples)

full_ds = concatenate_datasets([mendeley_ds, grazpedwri_ds_full]).shuffle(seed=42)

print(full_ds)
print(pd.Series(full_ds["source"]).value_counts())
print(pd.Series(full_ds["fracture_visible"]).value_counts())

Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 22711
})
grazpedwri    20327
mendeley       2384
Name: count, dtype: int64
True     15934
False     6777
Name: count, dtype: int64


In [ ]:
df_full = full_ds.to_pandas()
df_full["strata"] = df_full["source"] + "_" + df_full["fracture_visible"].astype(str)
print(df_full["strata"].value_counts())

strata
grazpedwri_True     13550
grazpedwri_False     6777
mendeley_True        2384
Name: count, dtype: int64


split 80/10/10 train/val/test, stratified by this combined strata column

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_full, test_size=0.2, stratify=df_full["strata"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["strata"], random_state=42
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print(train_df["strata"].value_counts())

Train: 18168 Val: 2271 Test: 2272
strata
grazpedwri_True     10840
grazpedwri_False     5421
mendeley_True        1907
Name: count, dtype: int64


In [ ]:
train_fracture = train_df[train_df["fracture_visible"] == True]
train_no_fracture = train_df[train_df["fracture_visible"] == False]

target_ratio = 1.5
target_fracture_count = int(len(train_no_fracture) * target_ratio)

train_fracture_balanced = train_fracture.sample(
    n=min(target_fracture_count, len(train_fracture)), random_state=42
)

train_df_balanced = pd.concat([train_fracture_balanced, train_no_fracture]).sample(frac=1, random_state=42)

print("Balanced train size:", len(train_df_balanced))
print(train_df_balanced["fracture_visible"].value_counts())
print(train_df_balanced["source"].value_counts())

Balanced train size: 13552
fracture_visible
True     8131
False    5421
Name: count, dtype: int64
source
grazpedwri    12342
mendeley       1210
Name: count, dtype: int64


All three splits into final HF datasets

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df_balanced.drop(columns="strata").reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.drop(columns="strata").reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.drop(columns="strata").reset_index(drop=True))

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 13552
})
Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 2271
})
Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 2272
})


Save splits to drive


In [ ]:
save_base = "/content/drive/MyDrive/Dissertation/datasets/processed_splits"
train_ds.save_to_disk(f"{save_base}/train_ds")
val_ds.save_to_disk(f"{save_base}/val_ds")
test_ds.save_to_disk(f"{save_base}/test_ds")

Saving the dataset (0/1 shards):   0%|          | 0/13552 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2271 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2272 [00:00<?, ? examples/s]

MEDGEMMA


Huggingface token

delete existing model----Delete this cell later

In [ ]:
import gc, torch

try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

install dependencies

In [ ]:
! pip install --upgrade --quiet bitsandbytes datasets evaluate peft tensorboard transformers trl torchao

Load the medgemma from huggingface

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-4b-it"

if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")

model_kwargs = dict(
    attn_implementation="sdpa",   # faster than "eager", built into PyTorch, no extra install needed
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Setup for finetuning

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

In [ ]:
from typing import Any
from PIL import Image

def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []
    for example in examples:
        image = Image.open(example["image_path"]).convert("RGB")
        images.append([image])

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": example["prompt"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": example["response"]},
                ],
            },
        ]

        texts.append(processor.apply_chat_template(
            messages, add_generation_prompt=False, tokenize=False
        ).strip())

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()

    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

Configure training parameters in an SFTConfig

In [ ]:
from trl import SFTConfig

num_train_epochs = 3
learning_rate = 2e-4

args = SFTConfig(
    output_dir="medgemma-4b-it-sft-lora-fracture",
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=50,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=learning_rate,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="linear",
    push_to_hub=False,
    report_to="tensorboard",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
)

Fine-tune the model

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds.shuffle(seed=42).select(range(200)),
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Launch the fine-tuning process.

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.748658,0.124808,0.106569,247390.000000,0.936512
100,0.110834,0.113957,0.109613,494262.000000,0.935807
150,0.106782,0.122250,0.092748,741277.000000,0.936896
200,0.105017,0.113311,0.102680,988755.000000,0.939714
250,0.103132,0.108755,0.104352,1236273.000000,0.939658
300,0.103008,0.109636,0.096930,1483085.000000,0.941911
350,0.101694,0.109843,0.105242,1730444.000000,0.939208
400,0.102322,0.110767,0.092939,1978046.000000,0.938441
450,0.099487,0.106479,0.109752,2225311.000000,0.942744
500,0.100417,0.105858,0.106114,2472717.000000,0.942484


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.748658,0.124808,0.106569,247390.000000,0.936512
100,0.110834,0.113957,0.109613,494262.000000,0.935807
150,0.106782,0.122250,0.092748,741277.000000,0.936896
200,0.105017,0.113311,0.102680,988755.000000,0.939714
250,0.103132,0.108755,0.104352,1236273.000000,0.939658
300,0.103008,0.109636,0.096930,1483085.000000,0.941911
350,0.101694,0.109843,0.105242,1730444.000000,0.939208
400,0.102322,0.110767,0.092939,1978046.000000,0.938441
450,0.099487,0.106479,0.109752,2225311.000000,0.942744
500,0.100417,0.105858,0.106114,2472717.000000,0.942484


TrainOutput(global_step=2541, training_loss=0.10533062363646717, metrics={'train_runtime': 22791.7382, 'train_samples_per_second': 1.784, 'train_steps_per_second': 0.111, 'total_flos': 3.384658679202029e+17, 'train_loss': 0.10533062363646717, 'epoch': 3.0})

**Incase the run get interrupted**

In [ ]:
#trainer.train(resume_from_checkpoint=True)

Check the loss curve

In [ ]:
print(trainer.state.log_history[-10:])

[{'loss': 0.08380699157714844, 'grad_norm': 0.41594770550727844, 'learning_rate': 1.511216056670602e-05, 'entropy': 0.08424072625115514, 'num_tokens': 11624499.0, 'mean_token_accuracy': 0.9557409280538559, 'epoch': 2.7744982290436835, 'step': 2350}, {'eval_loss': 0.09487459063529968, 'eval_runtime': 27.5219, 'eval_samples_per_second': 7.267, 'eval_steps_per_second': 1.817, 'eval_entropy': 0.08926354676485061, 'eval_num_tokens': 11624499.0, 'eval_mean_token_accuracy': 0.9503645193576813, 'epoch': 2.7744982290436835, 'step': 2350}, {'loss': 0.08524831771850586, 'grad_norm': 0.42775648832321167, 'learning_rate': 1.1176702085792996e-05, 'entropy': 0.08452757392078639, 'num_tokens': 11871688.0, 'mean_token_accuracy': 0.9546706950664521, 'epoch': 2.833530106257379, 'step': 2400}, {'eval_loss': 0.09437751770019531, 'eval_runtime': 27.7363, 'eval_samples_per_second': 7.211, 'eval_steps_per_second': 1.803, 'eval_entropy': 0.0903749418258667, 'eval_num_tokens': 11871688.0, 'eval_mean_token_accur

Save the finetuned model

In [ ]:
trainer.save_model("medgemma-4b-it-sft-lora-fracture-final")
processor.save_pretrained("medgemma-4b-it-sft-lora-fracture-final")

!mkdir -p /content/drive/MyDrive/Dissertation/models
!cp -r medgemma-4b-it-sft-lora-fracture-final /content/drive/MyDrive/Dissertation/models/

In [ ]:
from PIL import Image
import torch

def generate_response(image_path, prompt, max_new_tokens=100):
    image = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

In [ ]:
def predicted_fracture(generated_text):
    text = generated_text.lower()
    if "no fracture" in text or "no significant abnormality" in text:
        return False
    if "fracture" in text:
        return True
    return None  # ambiguous / couldn't parse — worth tracking separately

In [ ]:
import pandas as pd

results = []
sample = test_ds.shuffle(seed=42).select(range(100))  # start small — full test set later

for i, example in enumerate(sample):
    gen_text = generate_response(example["image_path"], example["prompt"])
    pred = predicted_fracture(gen_text)
    results.append({
        "image_path": example["image_path"],
        "source": example["source"],
        "ground_truth": example["fracture_visible"],
        "prediction": pred,
        "generated_text": gen_text
    })
    if i % 10 == 0:
        print(f"{i}/100 done")

results_df = pd.DataFrame(results)

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# drop unparseable predictions for now, but report how many there were
valid = results_df.dropna(subset=["prediction"])
print(f"Unparseable predictions: {len(results_df) - len(valid)} / {len(results_df)}")

acc = accuracy_score(valid["ground_truth"], valid["prediction"])
prec = precision_score(valid["ground_truth"], valid["prediction"])
rec = recall_score(valid["ground_truth"], valid["prediction"])
f1 = f1_score(valid["ground_truth"], valid["prediction"])

print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1: {f1:.3f}")
print(confusion_matrix(valid["ground_truth"], valid["prediction"]))

Unparseable predictions: 0 / 100
Accuracy: 0.640
Precision: 0.821
Recall: 0.639
F1: 0.719
[[18 10]
 [26 46]]


In [ ]:
results_df.to_csv("/content/drive/MyDrive/Dissertation/results/finetuned_results_100.csv", index=False)

In [ ]:
for src in valid["source"].unique():
    sub = valid[valid["source"] == src]
    print(f"\n--- {src} ---")
    print(f"n={len(sub)}")
    print(f"Accuracy: {accuracy_score(sub['ground_truth'], sub['prediction']):.3f}")

Check for later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/Dissertation/results/

Mounted at /content/drive
finetuned_results_100.csv


In [ ]:
from datasets import load_from_disk
save_base = "/content/drive/MyDrive/Dissertation/datasets/processed_splits"
train_ds = load_from_disk(f"{save_base}/train_ds")
val_ds = load_from_disk(f"{save_base}/val_ds")
test_ds = load_from_disk(f"{save_base}/test_ds")
print(train_ds.num_rows, val_ds.num_rows, test_ds.num_rows)

13552 2271 2272


In [ ]:
!nvidia-smi

Sat Aug 15 13:01:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

let check the finetune model, the 64% is small

In [ ]:
import pandas as pd
finetuned_results = pd.read_csv("/content/drive/MyDrive/Dissertation/results/finetuned_results_100.csv")
print(finetuned_results.shape)
finetuned_results.head()

(100, 5)


,image_path,source,ground_truth,prediction,generated_text
0,/content/drive/MyDrive/Dissertation/datasets/G...,grazpedwri,False,True,This L 2 pediatric wrist X-ray shows a fractur...
1,/content/images_part4/4947_0598075324_01_WRI-R...,grazpedwri,True,False,This R 2 pediatric wrist X-ray shows no fractu...
2,/content/images_part2/1911_0536899370_01_WRI-R...,grazpedwri,True,False,This R 2 pediatric wrist X-ray shows no fractu...
3,/content/drive/MyDrive/Dissertation/datasets/B...,mendeley,True,True,This X-ray shows evidence of a simple (non-dis...
4,/content/images_part4/6060_0589184569_06_WRI-L...,grazpedwri,True,False,This R 1 pediatric wrist X-ray shows no fractu...


In [ ]:
false_negatives = finetuned_results[
    (finetuned_results["ground_truth"] == True) & (finetuned_results["prediction"] == False)
]
print(f"{len(false_negatives)} false negatives")

for i, row in false_negatives.head(5).iterrows():
    print(f"\nSource: {row['source']}")
    print(f"Generated: {row['generated_text']}")

26 false negatives

Source: grazpedwri
Generated: This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.

Source: grazpedwri
Generated: This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.

Source: grazpedwri
Generated: This R 1 pediatric wrist X-ray shows no fracture or significant abnormality.

Source: grazpedwri
Generated: This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.

Source: grazpedwri
Generated: This R 1 pediatric wrist X-ray shows no fracture or significant abnormality.


In [ ]:
print(false_negatives["source"].value_counts())

# also check overall per-source accuracy from your original 100-sample results
for src in finetuned_results["source"].unique():
    sub = finetuned_results[finetuned_results["source"] == src]
    from sklearn.metrics import accuracy_score
    print(f"\n{src}: n={len(sub)}")
    print(f"Accuracy: {accuracy_score(sub['ground_truth'], sub['prediction']):.3f}")

source
grazpedwri    26
Name: count, dtype: int64

grazpedwri: n=86
Accuracy: 0.581

mendeley: n=14
Accuracy: 1.000


Load the basemodel Medgemma


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

base_model_id = "google/medgemma-4b-it"

base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_processor = AutoProcessor.from_pretrained(base_model_id)
base_processor.tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
from PIL import Image

def generate_response(image_path, prompt, model, processor, max_new_tokens=100):
    image = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def predicted_fracture(generated_text):
    text = generated_text.lower()
    if "no fracture" in text or "no significant abnormality" in text:
        return False
    if "fracture" in text:
        return True
    return None

In [ ]:

sample = test_ds.shuffle(seed=42).select(range(100))

base_results = []
for i, example in enumerate(sample):
    gen_text = generate_response(example["image_path"], example["prompt"], base_model, base_processor)
    pred = predicted_fracture(gen_text)
    base_results.append({
        "image_path": example["image_path"],
        "source": example["source"],
        "ground_truth": example["fracture_visible"],
        "prediction": pred,
        "generated_text": gen_text
    })
    if i % 10 == 0:
        print(f"{i}/100 done")

base_results_df = pd.DataFrame(base_results)
base_results_df.to_csv("/content/drive/MyDrive/Dissertation/results/base_results_100.csv", index=False)

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done


For base model

In [ ]:
base_results_df = pd.DataFrame(base_results)
base_results_df.to_csv("/content/drive/MyDrive/Dissertation/results/base_results_100.csv", index=False)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

valid_base = base_results_df.dropna(subset=["prediction"])
print(f"Unparseable predictions: {len(base_results_df) - len(valid_base)} / {len(base_results_df)}")

acc = accuracy_score(valid_base["ground_truth"], valid_base["prediction"])
prec = precision_score(valid_base["ground_truth"], valid_base["prediction"])
rec = recall_score(valid_base["ground_truth"], valid_base["prediction"])
f1 = f1_score(valid_base["ground_truth"], valid_base["prediction"])

print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1: {f1:.3f}")
print(confusion_matrix(valid_base["ground_truth"], valid_base["prediction"]))

Unparseable predictions: 0 / 100
Accuracy: 0.720
Precision: 0.720
Recall: 1.000
F1: 0.837
[[ 0 28]
 [ 0 72]]


Exploring Grazped for better Accuracy


In [ ]:
!unzip -l "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/folder_structure.zip" | head -30

Archive:  /content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/folder_structure.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2022-02-17 22:59   notebooks/
     5811  2021-01-04 21:18   notebooks/annotation_preview.ipynb
     3876  2021-01-06 18:56   notebooks/copy_files_by_csv.ipynb
     7582  2021-01-04 21:19   notebooks/image_conversion.ipynb
        0  2022-03-02 21:41   pascalvoc/
      629  2022-03-02 21:32   pascalvoc/0001_1297860395_01_WRI-L1_M014.xml
     1276  2022-03-02 21:32   pascalvoc/0001_1297860435_01_WRI-L2_M014.xml
      632  2022-03-02 21:32   pascalvoc/0002_0354485735_01_WRI-R1_F012.xml
      632  2022-03-02 21:32   pascalvoc/0002_0354485759_01_WRI-R2_F012.xml
     1273  2022-03-02 21:32   pascalvoc/0003_0662359226_01_WRI-R1_M011.xml
     1274  2022-03-02 21:32   pascalvoc/0003_0662359351_01_WRI-R2_M011.xml
     1272  2022-03-02 21:32   pascalvoc/0003_0663715732_02_WRI-R1_M011.xml
      955  2022-03-02 21:32   pascalvoc/0

In [ ]:
import zipfile

with zipfile.ZipFile("/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/folder_structure.zip") as z:
    names = z.namelist()
    top_level_dirs = set(n.split("/")[0] for n in names if "/" in n)
    print(top_level_dirs)

{'supervisely', 'notebooks', 'yolov5', 'pascalvoc'}


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/folder_structure.zip"
extract_to = "/content/yolov5_labels"

with zipfile.ZipFile(zip_path) as z:
    members = [n for n in z.namelist() if n.startswith("yolov5/")]
    z.extractall(path=extract_to, members=members)

print(f"Extracted {len(members)} files")

Extracted 20331 files


In [ ]:
import os

for root, dirs, files in os.walk("/content/yolov5_labels/yolov5"):
    print(root, "dirs:", dirs, "files sample:", files[:3])

/content/yolov5_labels/yolov5 dirs: ['labels', 'images'] files sample: ['meta.yaml']
/content/yolov5_labels/yolov5/labels dirs: [] files sample: ['2429_0839911355_03_WRI-L1_M009.txt', '2265_0947350428_01_WRI-L2_M014.txt', '5986_1287687508_01_WRI-R1_M016.txt']
/content/yolov5_labels/yolov5/images dirs: [] files sample: []


In [ ]:
# check meta.yaml first - this should list the class names in the correct index order
with open("/content/yolov5_labels/yolov5/meta.yaml") as f:
    print(f.read())

names:
- boneanomaly
- bonelesion
- foreignbody
- fracture
- metal
- periostealreaction
- pronatorsign
- softtissue
- text
nc: 9
path: 'FILL IN'
train: 'FILL IN'
val: 'FILL IN'
test: 'FILL IN'



In [ ]:
# now look at a few actual label files
sample_dir = "/content/yolov5_labels/yolov5/labels"
sample_files = ["2429_0839911355_03_WRI-L1_M009.txt", "2265_0947350428_01_WRI-L2_M014.txt", "5986_1287687508_01_WRI-R1_M016.txt"]

for fname in sample_files:
    print(f"\n--- {fname} ---")
    with open(os.path.join(sample_dir, fname)) as f:
        content = f.read()
        print(content if content.strip() else "(empty file — no annotations, likely no findings)")


--- 2429_0839911355_03_WRI-L1_M009.txt ---
8 0.095105 0.244652 0.027972 0.034759
8 0.084615 0.053476 0.085315 0.106952
3 0.53007 0.559492 0.268531 0.092246

--- 2265_0947350428_01_WRI-L2_M014.txt ---
8 0.073171 0.410714 0.109756 0.059524

--- 5986_1287687508_01_WRI-R1_M016.txt ---
8 0.901471 0.892857 0.047059 0.036204
4 0.444118 0.47407 0.132353 0.14775
3 0.400735 0.469667 0.145588 0.060665


In [ ]:
def yolo_box_to_location(x_center, y_center):
    # vertical position: wrist X-rays are typically oriented distal at top/bottom depending on projection
    # keep it simple/relative for now, refine if needed
    vertical = "distal" if y_center < 0.5 else "proximal"
    horizontal = "radial" if x_center < 0.5 else "ulnar"
    return f"{vertical} {horizontal} region"

def parse_yolo_label(filestem, labels_dir="/content/yolov5_labels/yolov5/labels"):
    path = os.path.join(labels_dir, f"{filestem}.txt")
    if not os.path.exists(path):
        return []  # no label file at all

    fracture_locations = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            if class_id == 3:  # fracture
                x_center, y_center = float(parts[1]), float(parts[2])
                fracture_locations.append(yolo_box_to_location(x_center, y_center))
    return fracture_locations

# test it
print(parse_yolo_label("2429_0839911355_03_WRI-L1_M009"))  # expect: ['proximal ulnar region']
print(parse_yolo_label("2265_0947350428_01_WRI-L2_M014"))  # expect: [] (no fracture)
print(parse_yolo_label("5986_1287687508_01_WRI-R1_M016"))  # expect: ['proximal radial region']

['proximal ulnar region']
[]
['distal radial region']


In [ ]:
import pandas as pd

df = pd.read_csv(Graz_path+"/dataset.csv")

def has_fracture_box(filestem, labels_dir="/content/yolov5_labels/yolov5/labels"):
    path = os.path.join(labels_dir, f"{filestem}.txt")
    if not os.path.exists(path):
        return None  # no label file at all - shouldn't happen given 20331 files extracted
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if parts and int(parts[0]) == 3:
                return True
    return False

df["yolo_has_fracture"] = df["filestem"].apply(has_fracture_box)

# cross-tab: does fracture_visible (NaN vs 1.0) line up with the real YOLO fracture box presence?
df["csv_fracture_visible"] = df["fracture_visible"].notna() & (df["fracture_visible"] == 1.0)

print(pd.crosstab(df["csv_fracture_visible"], df["yolo_has_fracture"]))

yolo_has_fracture     False  True 
csv_fracture_visible              
False                  6777      0
True                      0  13550


Rebuild Grazped data, added the yolo this time around


In [ ]:
def yolo_box_to_location(x_center, y_center):
    vertical = "distal" if y_center < 0.5 else "proximal"
    horizontal = "radial" if x_center < 0.5 else "ulnar"
    return f"{vertical} {horizontal} region"

def get_fracture_locations(filestem, labels_dir="/content/yolov5_labels/yolov5/labels"):
    path = os.path.join(labels_dir, f"{filestem}.txt")
    if not os.path.exists(path):
        return []
    locations = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if parts and int(parts[0]) == 3:
                x_center, y_center = float(parts[1]), float(parts[2])
                locations.append(yolo_box_to_location(x_center, y_center))
    return locations

def row_to_example_v2(row):
    img_path = row["image_path"]
    view = row['projection'] if pd.notna(row['projection']) else ""
    side = row['laterality'] if pd.notna(row['laterality']) else ""
    view_desc = f"{side} {view}".strip() or "wrist"

    findings = []
    has_fracture = pd.notna(row['fracture_visible']) and row['fracture_visible'] == 1.0

    if has_fracture:
        locations = get_fracture_locations(row["filestem"])
        if locations:
            loc_text = " and ".join(set(locations))  # dedupe if multiple boxes in same region
            ao = f" (AO classification: {row['ao_classification']})" if pd.notna(row.get('ao_classification')) else ""
            findings.append(f"a fracture in the {loc_text}{ao}")
        else:
            findings.append("a fracture")  # fallback, shouldn't happen given 100% match above

    if pd.notna(row.get('osteopenia')) and row['osteopenia'] == 1.0:
        findings.append("osteopenia")
    if pd.notna(row.get('metal')) and row['metal'] == 1.0:
        findings.append("metal hardware")
    if pd.notna(row.get('cast')) and row['cast'] == 1.0:
        findings.append("a cast in place")

    uncertain = pd.notna(row.get('diagnosis_uncertain')) and row['diagnosis_uncertain'] == 1.0

    if findings:
        response = f"This {view_desc} pediatric wrist X-ray shows " + ", ".join(findings) + "."
        if uncertain:
            response += " The diagnosis is noted as uncertain."
    else:
        response = f"This {view_desc} pediatric wrist X-ray shows no fracture or significant abnormality."

    return {
        "image_path": img_path,
        "prompt": "Describe any fracture or abnormal findings in this pediatric wrist X-ray.",
        "response": response,
        "fracture_visible": has_fracture,
        "source": "grazpedwri"
    }

In [ ]:
test_rows = df_matched.head(5)
for _, row in test_rows.iterrows():
    ex = row_to_example_v2(row)
    print(ex["response"])

This L 1 pediatric wrist X-ray shows no fracture or significant abnormality.
This L 2 pediatric wrist X-ray shows a fracture in the proximal ulnar region (AO classification: 23r-M/2.1).
This R 1 pediatric wrist X-ray shows no fracture or significant abnormality.
This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.
This R 1 pediatric wrist X-ray shows a fracture in the proximal ulnar region and proximal radial region (AO classification: 23-M/3.1).


In [ ]:
examples_full_v2 = df_matched.apply(row_to_example_v2, axis=1).tolist()
print(f"Total: {len(examples_full_v2)}")

# spot check a few more, including multi-fracture cases
import random
for ex in random.sample(examples_full_v2, 5):
    print(ex["response"])

Total: 20327
This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.
This R 2 pediatric wrist X-ray shows a fracture in the distal ulnar region (AO classification: 23r-M/3.1; 23u-E/7), a cast in place.
This L 1 pediatric wrist X-ray shows a fracture in the distal radial region (AO classification: 23-M/2.1), a cast in place.
This L 1 pediatric wrist X-ray shows a fracture in the proximal ulnar region and proximal radial region (AO classification: 23-M/2.1).
This R 2 pediatric wrist X-ray shows no fracture or significant abnormality.


REmerge

In [ ]:
from datasets import Dataset, concatenate_datasets

grazpedwri_ds_v2 = Dataset.from_list(examples_full_v2)
# mendeley_ds should still exist from earlier, or reload if needed

full_ds_v2 = concatenate_datasets([mendeley_ds, grazpedwri_ds_v2]).shuffle(seed=42)
print(full_ds_v2)

Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 22711
})


Redo stratified


In [ ]:
df_full_v2 = full_ds_v2.to_pandas()
df_full_v2["strata"] = df_full_v2["source"] + "_" + df_full_v2["fracture_visible"].astype(str)

from sklearn.model_selection import train_test_split

train_df_v2, temp_df_v2 = train_test_split(df_full_v2, test_size=0.2, stratify=df_full_v2["strata"], random_state=42)
val_df_v2, test_df_v2 = train_test_split(temp_df_v2, test_size=0.5, stratify=temp_df_v2["strata"], random_state=42)

print("Train:", len(train_df_v2), "Val:", len(val_df_v2), "Test:", len(test_df_v2))

Train: 18168 Val: 2271 Test: 2272


use a milder ratio

In [ ]:
train_fracture_v2 = train_df_v2[train_df_v2["fracture_visible"] == True]
train_no_fracture_v2 = train_df_v2[train_df_v2["fracture_visible"] == False]

target_ratio = 2.0  # gentler than before (was 1.5)
target_count = int(len(train_no_fracture_v2) * target_ratio)

train_fracture_balanced_v2 = train_fracture_v2.sample(n=min(target_count, len(train_fracture_v2)), random_state=42)
train_df_balanced_v2 = pd.concat([train_fracture_balanced_v2, train_no_fracture_v2]).sample(frac=1, random_state=42)

print("Balanced train size:", len(train_df_balanced_v2))
print(train_df_balanced_v2["fracture_visible"].value_counts())

Balanced train size: 16263
fracture_visible
True     10842
False     5421
Name: count, dtype: int64


Version 2 split

In [ ]:
from datasets import Dataset

train_ds_v2 = Dataset.from_pandas(train_df_balanced_v2.drop(columns="strata").reset_index(drop=True))
val_ds_v2 = Dataset.from_pandas(val_df_v2.drop(columns="strata").reset_index(drop=True))
test_ds_v2 = Dataset.from_pandas(test_df_v2.drop(columns="strata").reset_index(drop=True))

print(train_ds_v2)
print(val_ds_v2)
print(test_ds_v2)

save_base = "/content/drive/MyDrive/Dissertation/datasets/processed_splits_v2"
train_ds_v2.save_to_disk(f"{save_base}/train_ds")
val_ds_v2.save_to_disk(f"{save_base}/val_ds")
test_ds_v2.save_to_disk(f"{save_base}/test_ds")

print("Saved v2 splits")

Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 16263
})
Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 2271
})
Dataset({
    features: ['image_path', 'prompt', 'response', 'fracture_visible', 'source'],
    num_rows: 2272
})


Saving the dataset (0/1 shards):   0%|          | 0/16263 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2271 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2272 [00:00<?, ? examples/s]

Saved v2 splits


Want to retrain the model again this time with Yolov5


Delete existing model

In [ ]:
import gc, torch

for var in ["trainer", "trainer_v2", "model", "base_model", "base_for_ft"]:
    try:
        exec(f"del {var}")
    except NameError:
        pass

gc.collect()
torch.cuda.empty_cache()